# M3 + M4 on Kaggle: full-pool edge baselines and evaluation

**Attach as input:** your private Kaggle dataset holding the M1 `processed/` folder, five files in all:
`data_manifest.json`, `wifi_mqtt_full.parquet`, `wifi_mqtt_dev.parquet`, `wifi_profiling.parquet`, `ble.parquet`.
Files are found by name anywhere under `/kaggle/input`, so the dataset slug does not matter. The raw CSV and the Zenodo files are not needed.

**Settings:** Internet *On*. Accelerator: *None* is fine; the XGBoost GPU option is in the M3 cell.
**Run:** *Save Version → Save & Run All (Commit)* so it keeps running with the browser closed.
The estimate is 1–2 h with 4 vCPU, because the t5 model trains on about 7.2M rows; this has not been timed on the full pool yet.

**Output:** `/kaggle/working/m3_m4_full_outputs.zip` contains `reports/m3/full/`, `reports/m4/full/` and the M3 and M4 run folders, with the models.
Download it and unzip it at the repo root. The partition is rebuilt here from the fixed seed.

In [ ]:
import os, subprocess, sys
REPO = "/kaggle/working/IDS-code"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Indrajit147/IDS-code.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
# --no-deps: keep Kaggle's preinstalled stack intact; versions are recorded per run.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO, "--no-deps"], check=True)
os.chdir(REPO)
os.environ["IDS_REPO_ROOT"] = REPO
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Rebuild the M2 partition for the full pool (deterministic, ~1 min)
!python scripts/m2_build_partitions.py --env kaggle --pools full

In [ ]:
# M3 full-pool run. With a GPU attached you can add:  --set models.xgb.device=cuda
!python scripts/m3_baselines.py --env kaggle --pool full

In [ ]:
# M4: reload the saved models, check them against M3's logged numbers, write the standard report
!python scripts/m4_evaluate.py --env kaggle --pool full


In [ ]:
import shutil, glob, os
stage = "/kaggle/working/m3_m4_stage"
shutil.rmtree(stage, ignore_errors=True)
for pool_report in ("reports/m3/full", "reports/m4/full"):
    shutil.copytree(pool_report, f"{stage}/{pool_report}")
for pattern in ("*_m3_baselines_full", "*_m4_evaluate_full"):
    runs = sorted(glob.glob(f"/kaggle/working/results/runs/{pattern}"))
    if runs:
        shutil.copytree(runs[-1], f"{stage}/results/runs/{os.path.basename(runs[-1])}")
shutil.make_archive("/kaggle/working/m3_m4_full_outputs", "zip", stage)
shutil.rmtree(stage)
print(open("reports/m4/full/EVALUATION.md", encoding="utf-8").read()[:4000])
